<a href="https://colab.research.google.com/github/hatamimatt/Climatematch_Academy_Course/blob/main/tutorials/W1D1_ClimateSystemOverview/student/W1D1_Tutorial1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 1: Creating DataArrays and Datasets to Assess Global Climate Data

**Week 1, Day 1, Climate System Overview**

**Content creators:** Sloane Garelick, Julia Kent

**Content reviewers:** Yosmely Bermúdez, Katrina Dobson, Younkap Nina Duplex, Danika Gupta, Maria Gonzalez, Will Gregory, Nahid Hasan, Paul Heubel, Sherry Mi, Beatriz Cosenza Muralles, Jenna Pearson, Chi Zhang, Ohad Zivan

**Content editors:** Paul Heubel, Jenna Pearson, Chi Zhang, Ohad Zivan

**Production editors:** Wesley Banfield, Paul Heubel, Jenna Pearson, Konstantine Tsafatinos, Chi Zhang, Ohad Zivan

**Our 2024 Sponsors:** NFDI4Earth and CMIP

## ![project pythia](https://github.com/neuromatch/climate-course-content/blob/main/tutorials/static/pythia-logo.svg?raw=true)

Pythia credit: Rose, B. E. J., Kent, J., Tyle, K., Clyne, J., Banihirwe, A., Camron, D., May, R., Grover, M., Ford, R. R., Paul, K., Morley, J., Eroglu, O., Kailyn, L., & Zacharias, A. (2023). Pythia Foundations (Version v2023.05.01) https://zenodo.org/record/8065851

## ![CMIP.png](https://github.com/ClimateMatchAcademy/course-content/blob/main/tutorials/Art/CMIP.png?raw=true)

# Tutorial Objectives

*Estimated timing of tutorial:* 20 minutes

As you just learned in the Introduction to Climate video, variations in global climate involve various forcings, feedbacks, and interactions between multiple processes and systems. Because of this complexity, global climate datasets are often very large with multiple dimensions and variables.

One useful computational tool for organizing, analyzing, and interpreting large global datasets is [Xarray](https://xarray.pydata.org/en/v2023.05.0/getting-started-guide/why-xarray.html), an open-source project and Python package that makes working with labeled multi-dimensional arrays simple and efficient.

In this first tutorial, we will use the `DataArray` and `Dataset` objects, which are used to represent and manipulate spatial data, to practice organizing large global climate datasets, and to understand variations in Earth's climate system.

# Setup

Similar to `numpy`, `np`; `pandas`, `pd`; you may often encounter `xarray` imported within a shortened namespace as `xr`.

In [1]:
# imports
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
# @title Install and import feedback gadget

!pip3 install vibecheck datatops --quiet

from vibecheck import DatatopsContentReviewContainer
def content_review(notebook_section: str):
    return DatatopsContentReviewContainer(
        "",  # No text prompt
        notebook_section,
        {
            "url": "https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab",
            "name": "comptools_4clim",
            "user_key": "l5jpxuee",
        },
    ).render()


feedback_prefix = "W1D1_T1"

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 64.6 MB/s eta 0:00:00


In [3]:
# @title Figure Settings
import ipywidgets as widgets  # interactive display

%config InlineBackend.figure_format = 'retina'
plt.style.use(
    "https://raw.githubusercontent.com/neuromatch/climate-course-content/main/cma.mplstyle"
)

In [4]:
# @title Video 1: Introduction to Climate

from ipywidgets import widgets
from IPython.display import YouTubeVideo
from IPython.display import IFrame
from IPython.display import display


class PlayVideo(IFrame):
    def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
        self.id = id
        if source == "Bilibili":
            src = f"https://player.bilibili.com/player.html?bvid={id}&page={page}"
        elif source == "Osf":
            src = f"https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render"
        super(PlayVideo, self).__init__(src, width, height, **kwargs)


def display_videos(video_ids, W=400, H=300, fs=1):
    tab_contents = []
    for i, video_id in enumerate(video_ids):
        out = widgets.Output()
        with out:
            if video_ids[i][0] == "Youtube":
                video = YouTubeVideo(
                    id=video_ids[i][1], width=W, height=H, fs=fs, rel=0
                )
                print(f"Video available at https://youtube.com/watch?v={video.id}")
            else:
                video = PlayVideo(
                    id=video_ids[i][1],
                    source=video_ids[i][0],
                    width=W,
                    height=H,
                    fs=fs,
                    autoplay=False,
                )
                if video_ids[i][0] == "Bilibili":
                    print(
                        f"Video available at https://www.bilibili.com/video/{video.id}"
                    )
                elif video_ids[i][0] == "Osf":
                    print(f"Video available at https://osf.io/{video.id}")
            display(video)
        tab_contents.append(out)
    return tab_contents


video_ids = [("Youtube", "mc-DkvYLdOA"), ("Bilibili", "BV1Th4y1j7SS")]
tab_contents = display_videos(video_ids, W=730, H=410)
tabs = widgets.Tab()
tabs.children = tab_contents
for i in range(len(tab_contents)):
    tabs.set_title(i, video_ids[i][0])
display(tabs)

In [5]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Intro_to_Climate_Video")

In [6]:
# @markdown
from ipywidgets import widgets
from IPython.display import IFrame

link_id = "4suf5"

print(f"If you want to download the slides: https://osf.io/download/{link_id}/")
IFrame(src=f"https://mfr.ca-1.osf.io/render?url=https://osf.io/{link_id}/?direct%26mode=render%26action=download%26mode=render", width=854, height=480)

If you want to download the slides: https://osf.io/download/4suf5/


In [7]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Intro_to_Climate_Slides")

# Introducing the `DataArray` and `Dataset`

[Xarray](https://xarray.pydata.org/en/v2023.05.0/getting-started-guide/why-xarray.html) expands on the capabilities on [NumPy](https://numpy.org/doc/stable/user/index.html#user) arrays, providing a lot of streamlined data manipulation. It is similar in that respect to [Pandas](https://pandas.pydata.org/docs/user_guide/index.html#user-guide), but whereas Pandas excels at working with tabular data, Xarray is focused on N-dimensional arrays of data (i.e. grids). Its interface is based largely on the netCDF data model (variables, attributes, and dimensions), but it goes beyond the traditional netCDF interfaces to provide functionality similar to netCDF-java's [Common Data Model (CDM)](https://docs.unidata.ucar.edu/netcdf-java/current/userguide/common_data_model_overview.html).

# Section 1: Creation of a `DataArray` Object

The `DataArray` is one of the basic building blocks of Xarray (see docs [here](http://xarray.pydata.org/en/stable/user-guide/data-structures.html#dataarray)). It provides a `numpy.ndarray`-like object that expands to provide two critical pieces of functionality:

1. Coordinate names and values are stored with the data, making slicing and indexing much more powerful
2. It has a built-in container for attributes

Here we'll initialize a `DataArray` object by wrapping a plain NumPy array, and explore a few of its properties.

## Section 1.1: Generate a Random Numpy Array

For our first example, we'll just [create a random array](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html) of "temperature" data in units of Kelvin:

In [8]:
rand_data = 283 + 5 * np.random.randn(5, 3, 4)
rand_data

array([[[282.04588007, 279.47576051, 280.4529317 , 284.32761729],
        [275.55113455, 283.80752585, 288.76903448, 277.98709693],
        [282.90861446, 282.6226961 , 280.89475845, 280.23708299]],

       [[283.7466599 , 285.79373708, 277.05802243, 279.3658904 ],
        [274.85393116, 285.90116556, 281.8340748 , 281.41236256],
        [277.11970889, 282.72208658, 290.76453459, 291.21281881]],

       [[282.48387308, 285.54479814, 279.43519542, 276.21947567],
        [290.0991832 , 282.32967154, 287.84098064, 294.30976957],
        [286.33271193, 279.22189056, 276.3084913 , 280.8153294 ]],

       [[293.03746   , 285.5966679 , 275.26195072, 273.52242365],
        [281.04975803, 273.38629139, 285.02705711, 284.36239693],
        [282.03944934, 281.05873889, 283.86814158, 271.69425898]],

       [[282.09431165, 289.57262979, 273.67560279, 289.2716462 ],
        [285.23406387, 275.37034847, 270.56669026, 286.27465671],
        [282.81394256, 283.90772837, 280.17537736, 284.20930733]]])

## Section 1.2: Wrap the Array: First Attempt

Now we create a basic `DataArray` just by passing our plain `data` as an input:

In [9]:
temperature = xr.DataArray(rand_data)
temperature

<xarray.DataArray (dim_0: 5, dim_1: 3, dim_2: 4)> Size: 480B
array([[[282.04588007, 279.47576051, 280.4529317 , 284.32761729],
        [275.55113455, 283.80752585, 288.76903448, 277.98709693],
        [282.90861446, 282.6226961 , 280.89475845, 280.23708299]],

       [[283.7466599 , 285.79373708, 277.05802243, 279.3658904 ],
        [274.85393116, 285.90116556, 281.8340748 , 281.41236256],
        [277.11970889, 282.72208658, 290.76453459, 291.21281881]],

       [[282.48387308, 285.54479814, 279.43519542, 276.21947567],
        [290.0991832 , 282.32967154, 287.84098064, 294.30976957],
        [286.33271193, 279.22189056, 276.3084913 , 280.8153294 ]],

       [[293.03746   , 285.5966679 , 275.26195072, 273.52242365],
        [281.04975803, 273.38629139, 285.02705711, 284.36239693],
        [282.03944934, 281.05873889, 283.86814158, 271.69425898]],

       [[282.09431165, 289.57262979, 273.67560279, 289.2716462 ],
        [285.23406387, 275.37034847, 270.56669026, 286.27465671],
        [282.81394256, 283.90772837, 280.17537736, 284.20930733]]])
Dimensions without coordinates: dim_0, dim_1, dim_2

Note two things:

1. Xarray generates some basic dimension names for us (`dim_0`, `dim_1`, `dim_2`). We'll improve this with better names in the next example.
2. Wrapping the numpy array in a `DataArray` gives us a rich display in the notebook! (Try clicking the array symbol to expand or collapse the view)

## Section 1.3: Assign Dimension Names

Much of the power of Xarray comes from making use of named dimensions. So let's add some more useful names! We can do that by passing an ordered list of names using the keyword argument `dims`:

In [10]:
temperature = xr.DataArray(rand_data, dims=["time", "lat", "lon"])
temperature

<xarray.DataArray (time: 5, lat: 3, lon: 4)> Size: 480B
array([[[282.04588007, 279.47576051, 280.4529317 , 284.32761729],
        [275.55113455, 283.80752585, 288.76903448, 277.98709693],
        [282.90861446, 282.6226961 , 280.89475845, 280.23708299]],

       [[283.7466599 , 285.79373708, 277.05802243, 279.3658904 ],
        [274.85393116, 285.90116556, 281.8340748 , 281.41236256],
        [277.11970889, 282.72208658, 290.76453459, 291.21281881]],

       [[282.48387308, 285.54479814, 279.43519542, 276.21947567],
        [290.0991832 , 282.32967154, 287.84098064, 294.30976957],
        [286.33271193, 279.22189056, 276.3084913 , 280.8153294 ]],

       [[293.03746   , 285.5966679 , 275.26195072, 273.52242365],
        [281.04975803, 273.38629139, 285.02705711, 284.36239693],
        [282.03944934, 281.05873889, 283.86814158, 271.69425898]],

       [[282.09431165, 289.57262979, 273.67560279, 289.2716462 ],
        [285.23406387, 275.37034847, 270.56669026, 286.27465671],
        [282.81394256, 283.90772837, 280.17537736, 284.20930733]]])
Dimensions without coordinates: time, lat, lon

This is already an improvement over a NumPy array because we have names for each of the dimensions (or axes). Even better, we can associate arrays representing the values for the coordinates for each of these dimensions with the data when we create the `DataArray`. We'll see this in the next example.

# Section 2: Create a `DataArray` with Named Coordinates

## Section 2.1: Make Time and Space Coordinates

Here we will use [Pandas](https://foundations.projectpythia.org/core/pandas) to create an array of [datetime data](https://foundations.projectpythia.org/core/datetime), which we will then use to create a `DataArray` with a named coordinate `time`.

In [11]:
times_index = pd.date_range("2018-01-01", periods=5)
times_index

DatetimeIndex(['2018-01-01', '2018-01-02', '2018-01-03', '2018-01-04',
               '2018-01-05'],
              dtype='datetime64[ns]', freq='D')

We'll also create arrays to represent sample longitude and latitude:

In [12]:
lons = np.linspace(-120, -60, 4)
lats = np.linspace(25, 55, 3)

### Section 2.1.1: Initialize the `DataArray` with Complete Coordinate Info

When we create the `DataArray` instance, we pass in the arrays we just created:

In [13]:
temperature = xr.DataArray(
    rand_data, coords=[times_index, lats, lons], dims=["time", "lat", "lon"]
)
temperature

<xarray.DataArray (time: 5, lat: 3, lon: 4)> Size: 480B
array([[[282.04588007, 279.47576051, 280.4529317 , 284.32761729],
        [275.55113455, 283.80752585, 288.76903448, 277.98709693],
        [282.90861446, 282.6226961 , 280.89475845, 280.23708299]],

       [[283.7466599 , 285.79373708, 277.05802243, 279.3658904 ],
        [274.85393116, 285.90116556, 281.8340748 , 281.41236256],
        [277.11970889, 282.72208658, 290.76453459, 291.21281881]],

       [[282.48387308, 285.54479814, 279.43519542, 276.21947567],
        [290.0991832 , 282.32967154, 287.84098064, 294.30976957],
        [286.33271193, 279.22189056, 276.3084913 , 280.8153294 ]],

       [[293.03746   , 285.5966679 , 275.26195072, 273.52242365],
        [281.04975803, 273.38629139, 285.02705711, 284.36239693],
        [282.03944934, 281.05873889, 283.86814158, 271.69425898]],

       [[282.09431165, 289.57262979, 273.67560279, 289.2716462 ],
        [285.23406387, 275.37034847, 270.56669026, 286.27465671],
        [282.81394256, 283.90772837, 280.17537736, 284.20930733]]])
Coordinates:
  * time     (time) datetime64[ns] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0

### Section 2.1.2: Set Useful Attributes

We can also set some attribute metadata, which will help provide clear descriptions of the data. In this case, we can specify that we're looking at 'air_temperature' data and the units are 'Kelvin'.

In [14]:
temperature.attrs["units"] = "Kelvin"
temperature.attrs["standard_name"] = "air_temperature"

temperature

<xarray.DataArray (time: 5, lat: 3, lon: 4)> Size: 480B
array([[[282.04588007, 279.47576051, 280.4529317 , 284.32761729],
        [275.55113455, 283.80752585, 288.76903448, 277.98709693],
        [282.90861446, 282.6226961 , 280.89475845, 280.23708299]],

       [[283.7466599 , 285.79373708, 277.05802243, 279.3658904 ],
        [274.85393116, 285.90116556, 281.8340748 , 281.41236256],
        [277.11970889, 282.72208658, 290.76453459, 291.21281881]],

       [[282.48387308, 285.54479814, 279.43519542, 276.21947567],
        [290.0991832 , 282.32967154, 287.84098064, 294.30976957],
        [286.33271193, 279.22189056, 276.3084913 , 280.8153294 ]],

       [[293.03746   , 285.5966679 , 275.26195072, 273.52242365],
        [281.04975803, 273.38629139, 285.02705711, 284.36239693],
        [282.03944934, 281.05873889, 283.86814158, 271.69425898]],

       [[282.09431165, 289.57262979, 273.67560279, 289.2716462 ],
        [285.23406387, 275.37034847, 270.56669026, 286.27465671],
        [282.81394256, 283.90772837, 280.17537736, 284.20930733]]])
Coordinates:
  * time     (time) datetime64[ns] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
Attributes:
    units:          Kelvin
    standard_name:  air_temperature

### Section 2.1.3: Attributes Are Not Preserved by Default!

Notice what happens if we perform a mathematical operaton with the `DataArray`: the coordinate values persist, but the attributes are lost. This is done because it is very challenging to know if the attribute metadata is still correct or appropriate after arbitrary arithmetic operations.

To illustrate this, we'll do a simple unit conversion from Kelvin to Celsius:

In [15]:
temperature_in_celsius = temperature - 273.15
temperature_in_celsius

<xarray.DataArray (time: 5, lat: 3, lon: 4)> Size: 480B
array([[[ 8.89588007,  6.32576051,  7.3029317 , 11.17761729],
        [ 2.40113455, 10.65752585, 15.61903448,  4.83709693],
        [ 9.75861446,  9.4726961 ,  7.74475845,  7.08708299]],

       [[10.5966599 , 12.64373708,  3.90802243,  6.2158904 ],
        [ 1.70393116, 12.75116556,  8.6840748 ,  8.26236256],
        [ 3.96970889,  9.57208658, 17.61453459, 18.06281881]],

       [[ 9.33387308, 12.39479814,  6.28519542,  3.06947567],
        [16.9491832 ,  9.17967154, 14.69098064, 21.15976957],
        [13.18271193,  6.07189056,  3.1584913 ,  7.6653294 ]],

       [[19.88746   , 12.4466679 ,  2.11195072,  0.37242365],
        [ 7.89975803,  0.23629139, 11.87705711, 11.21239693],
        [ 8.88944934,  7.90873889, 10.71814158, -1.45574102]],

       [[ 8.94431165, 16.42262979,  0.52560279, 16.1216462 ],
        [12.08406387,  2.22034847, -2.58330974, 13.12465671],
        [ 9.66394256, 10.75772837,  7.02537736, 11.05930733]]])
Coordinates:
  * time     (time) datetime64[ns] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
Attributes:
    units:          Kelvin
    standard_name:  air_temperature

We usually wish to keep metadata with our dataset, even after manipulating the data. For example it can tell us what the units are of a variable of interest. So when you perform operations on your data, make sure to check that all the information you want is carried over. If it isn't, you can add it back in following the instructions in the section before this. For an in-depth discussion of how Xarray handles metadata, you can find more information in the Xarray documents [here](http://xarray.pydata.org/en/stable/getting-started-guide/faq.html#approach-to-metadata).

# Section 3: The `Dataset`: a Container for `DataArray`s with Shared Coordinates

Along with `DataArray`, the other key object type in Xarray is the `Dataset`, which is a dictionary-like container that holds one or more `DataArray`s, which can also optionally share coordinates (see docs [here](http://xarray.pydata.org/en/stable/user-guide/data-structures.html#dataset)).

The most common way to create a `Dataset` object is to load data from a file (which we will practice in a later tutorial). Here, instead, we will create another `DataArray` and combine it with our `temperature` data.

This will illustrate how the information about common coordinate axes is used.

## Section 3.1: Create a Pressure `DataArray` Using the Same Coordinates

For our next `DataArry` example, we'll create a random array of `pressure` data in units of hectopascal (hPa). This code mirrors how we created the `temperature` object above.

In [16]:
pressure_data = 1000.0 + 5 * np.random.randn(5, 3, 4)
pressure = xr.DataArray(
    pressure_data, coords=[times_index, lats, lons], dims=["time", "lat", "lon"]
)
pressure.attrs["units"] = "hPa"
pressure.attrs["standard_name"] = "air_pressure"

pressure

<xarray.DataArray (time: 5, lat: 3, lon: 4)> Size: 480B
array([[[ 994.54143065, 1004.52400986, 1010.47117715, 1007.89144176],
        [ 999.67203202,  995.74342626,  998.0000639 ,  998.7516963 ],
        [1002.21481632,  998.31933249, 1007.96696324, 1001.49262674]],

       [[1004.19109277, 1007.50675439,  999.83389297, 1001.86939433],
        [ 997.91870834,  999.15113347, 1000.92924086,  993.04862113],
        [ 985.57102589, 1007.23310941,  992.89330067, 1006.41988112]],

       [[1000.91806538,  996.5241368 , 1003.96973713, 1001.96506268],
        [ 999.22005065, 1002.96344197, 1001.14605606,  998.93629594],
        [1000.81400616,  996.47313157, 1005.60952197, 1000.81732065]],

       [[1000.98742889,  995.51791385,  995.31264487, 1008.44157572],
        [1002.08096478,  997.15884935, 1004.2910162 ,  997.24487562],
        [ 997.16629075, 1000.46907375, 1002.49792232,  998.99904714]],

       [[1008.67613154, 1003.08044514,  995.24176476,  999.63124509],
        [1010.59688133,  998.93964805,  998.54592603, 1003.60228498],
        [ 995.00072261, 1001.45228891, 1002.11270071, 1005.29293454]]])
Coordinates:
  * time     (time) datetime64[ns] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
Attributes:
    units:          hPa
    standard_name:  air_pressure

## Section 3.2: Create a `Dataset` Object

Each `DataArray` in our `Dataset` needs a name!

The most straightforward way to create a `Dataset` with our `temperature` and `pressure` arrays is to pass a dictionary using the keyword argument `data_vars`:

In [17]:
ds = xr.Dataset(data_vars={"Temperature": temperature, "Pressure": pressure})
ds

<xarray.Dataset> Size: 1kB
Dimensions:      (time: 5, lat: 3, lon: 4)
Coordinates:
  * time         (time) datetime64[ns] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat          (lat) float64 24B 25.0 40.0 55.0
  * lon          (lon) float64 32B -120.0 -100.0 -80.0 -60.0
Data variables:
    Temperature  (time, lat, lon) float64 480B 282.0 279.5 280.5 ... 280.2 284.2
    Pressure     (time, lat, lon) float64 480B 994.5 1.005e+03 ... 1.005e+03

Notice that the `Dataset` object `ds` is aware that both data arrays sit on the same coordinate axes.

## Section 3.3: Access Data Variables and Coordinates in a `Dataset`

We can pull out any of the individual `DataArray` objects in a few different ways.

Using the "dot" notation:

In [18]:
ds.Pressure

<xarray.DataArray 'Pressure' (time: 5, lat: 3, lon: 4)> Size: 480B
array([[[ 994.54143065, 1004.52400986, 1010.47117715, 1007.89144176],
        [ 999.67203202,  995.74342626,  998.0000639 ,  998.7516963 ],
        [1002.21481632,  998.31933249, 1007.96696324, 1001.49262674]],

       [[1004.19109277, 1007.50675439,  999.83389297, 1001.86939433],
        [ 997.91870834,  999.15113347, 1000.92924086,  993.04862113],
        [ 985.57102589, 1007.23310941,  992.89330067, 1006.41988112]],

       [[1000.91806538,  996.5241368 , 1003.96973713, 1001.96506268],
        [ 999.22005065, 1002.96344197, 1001.14605606,  998.93629594],
        [1000.81400616,  996.47313157, 1005.60952197, 1000.81732065]],

       [[1000.98742889,  995.51791385,  995.31264487, 1008.44157572],
        [1002.08096478,  997.15884935, 1004.2910162 ,  997.24487562],
        [ 997.16629075, 1000.46907375, 1002.49792232,  998.99904714]],

       [[1008.67613154, 1003.08044514,  995.24176476,  999.63124509],
        [1010.59688133,  998.93964805,  998.54592603, 1003.60228498],
        [ 995.00072261, 1001.45228891, 1002.11270071, 1005.29293454]]])
Coordinates:
  * time     (time) datetime64[ns] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
Attributes:
    units:          hPa
    standard_name:  air_pressure

... or using dictionary access like this:

In [19]:
ds["Pressure"]

<xarray.DataArray 'Pressure' (time: 5, lat: 3, lon: 4)> Size: 480B
array([[[ 994.54143065, 1004.52400986, 1010.47117715, 1007.89144176],
        [ 999.67203202,  995.74342626,  998.0000639 ,  998.7516963 ],
        [1002.21481632,  998.31933249, 1007.96696324, 1001.49262674]],

       [[1004.19109277, 1007.50675439,  999.83389297, 1001.86939433],
        [ 997.91870834,  999.15113347, 1000.92924086,  993.04862113],
        [ 985.57102589, 1007.23310941,  992.89330067, 1006.41988112]],

       [[1000.91806538,  996.5241368 , 1003.96973713, 1001.96506268],
        [ 999.22005065, 1002.96344197, 1001.14605606,  998.93629594],
        [1000.81400616,  996.47313157, 1005.60952197, 1000.81732065]],

       [[1000.98742889,  995.51791385,  995.31264487, 1008.44157572],
        [1002.08096478,  997.15884935, 1004.2910162 ,  997.24487562],
        [ 997.16629075, 1000.46907375, 1002.49792232,  998.99904714]],

       [[1008.67613154, 1003.08044514,  995.24176476,  999.63124509],
        [1010.59688133,  998.93964805,  998.54592603, 1003.60228498],
        [ 995.00072261, 1001.45228891, 1002.11270071, 1005.29293454]]])
Coordinates:
  * time     (time) datetime64[ns] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
Attributes:
    units:          hPa
    standard_name:  air_pressure

We'll return to the `Dataset` object when we start loading data from files in later tutorials today.

# Summary

In this initial tutorial, the `DataArray` and `Dataset` objects were utilized to create and explore synthetic examples of climate data.

# Resources

Code and data for this tutorial is based on existing content from [Project Pythia](https://foundations.projectpythia.org/core/xarray/xarray-intro).